# 03 Post Figures - Bahn Delay Story

Objective: create polished charts for the portfolio writeup and dashboard screenshots.

Run this after `02_analysis.ipynb` has identified the final story.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))

from bahn_delay_story.config import FIGURES_DIR, PROCESSED_DIR
from bahn_delay_story.plots import hourly_heatmap, train_type_late_share

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 80)
FIGURES_DIR


## Load Final Tables

In [ ]:
with duckdb.connect() as con:
    train_type_day = con.execute(
        "SELECT * FROM read_parquet($path)",
        {"path": str(PROCESSED_DIR / "train_type_day_metrics.parquet")},
    ).df()
    hourly = con.execute(
        "SELECT * FROM read_parquet($path)",
        {"path": str(PROCESSED_DIR / "hourly_delay_metrics.parquet")},
    ).df()

train_type_day.head()


## Headline Figure

In [ ]:
headline_types = [value for value in ["ICE", "IC", "EC", "RE", "RB", "S"] if value in train_type_day["train_type"].unique()]
headline_data = train_type_day[train_type_day["train_type"].isin(headline_types)].copy()

fig = train_type_late_share(headline_data)
fig.update_layout(
    title="Late Share By Train Type",
    template="plotly_white",
    height=520,
)
fig.write_html(FIGURES_DIR / "late_share_by_train_type.html")
fig


## Hourly Pattern Figure

In [ ]:
hourly_subset = hourly[hourly["train_type"].isin(headline_types)].copy()
fig = hourly_heatmap(hourly_subset)
fig.update_layout(
    title="Average Late Share By Weekday And Hour",
    template="plotly_white",
    height=520,
)
fig.write_html(FIGURES_DIR / "hourly_late_share_heatmap.html")
fig


## Export Notes

Plotly HTML files are saved in `reports/figures/`. For static PNG export, add `kaleido` later and call `fig.write_image(...)`.